# Demo 2 — Più media significa più attenuazione ovunque?

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 44100
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
fig = go.Figure()
for L in (2, 8, 32):
    b_media = np.ones(L) / L
    f_media, H_media = signal.freqz(b_media, worN=4096, fs=SR)
    fig.add_trace(go.Scatter(x=f_media, y=al.to_db(np.abs(H_media), floor=1e-6),
                            name=f"L = {L}, ordine {L-1}"))
fig.update_xaxes(title_text="Frequenza [Hz]", range=[0, SR/2])
fig.update_yaxes(title_text="Guadagno [dB]", range=[-70, 3])
fig.update_layout(height=380, title="Media mobile: lobo principale e lobi laterali")
fig.show()
chitarra, sr = al.load_audio(AUDIO / "Guitar_C4.wav", sr=SR, seconds=4.0)
versioni_media = [("Originale", np.pad(chitarra, (0, 31)))]
for L in (2, 8, 32):
    filtrata = al.filter_signal(np.ones(L)/L, [1.], np.pad(chitarra, (0, 31)))
    versioni_media.append((f"Media su {L} campioni", filtrata))
al.compare_audio(versioni_media, sr)
